# NLP Pipeline Setup & Usage (Commented)

This notebook shows how to install, configure, and exercise the `nlp_pipeline` package.
Inline comments were added to code cells to explain what each block is doing and why.


# NLP Pipeline Notebook

This notebook walks through **installation**, **setup/configuration**, and **usage** of the `nlp_project` NLP pipeline.

Stages covered:
1. Data Preparation: segmentation, tokenization, stop-word removal
2. AI Prepping: optional stemming, lemmatization, POS tagging, NER
3. AI Training: TF-IDF + TensorFlow/Keras baseline classifier


## 1) Environment Setup
Run from repo root (`/home/connor/ECE495_IndepStudy/Content/Code`) in your `.venv`.


In [ ]:
# Check Python version and working directory

import sys
print('Python:', sys.version) 

import os
print(os.getcwd())
print(os.listdir("."))

Python: 3.13.7 (main, Aug 15 2025, 12:34:02) [GCC 15.2.1 20250813]
/home/connor/ECE495_IndepStudy/Content/Code/nlp_project
['nlp_project.egg-info', 'tests', 'scratch.csv', 'README.md', 'nlp_pipeline', 'NLP_Pipeline_Setup_Usage.ipynb', 'pyproject.toml', '.pytest_cache']


In [ ]:
# Install the nlp_project package in editable mode

%pip install -e ../nlp_project

Obtaining file:///home/connor/ECE495_IndepStudy/Content/Code/nlp_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nlp_project (pyproject.toml) ... done
  Created wheel for nlp_project: filename=nlp_project-0.1.0-0.editable-py3-none-any.whl size=3997 sha256=171fe34719376b7688a7e4c217dd6108e7b89167e51aec5bc2820fc7dc5b2d3f
  Stored in directory: /tmp/pip-ephem-wheel-cache-0etujwh2/wheels/34/d6/4e/fe1ab67262d872a1d6c76a4ca56fe33f2e797b8a932745fe36
Successfully built nlp_project
  Attempting uninstall: nlp_project
    Found existing installation: nlp_project 0.1.0
    Uninstalling nlp_project-0.1.0:
      Successfully uninstalled nlp_project-0.1.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Download spaCy English model (required for lemmatization and NER)

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 3.3 MB/s  0:00:04 eta 0:00:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 2) Imports and Deterministic Config


In [ ]:
# Import core dependencies
# pandas/numpy for data manipulation; pipeline modules for NLP processing stages
import pandas as pd 
import numpy as np
from nlp_pipeline.segmentation import segment_sentences
from nlp_pipeline.tokenizing import tokenize_text
from nlp_pipeline.stopwords import remove_stopwords
from nlp_pipeline.normalize import normalize_text, lemmatize_text, stem_tokens
from nlp_pipeline.tagging import pos_tag_text, named_entities
from nlp_pipeline.vectorize import TfidfTextVectorizer
from nlp_pipeline.tf_model import set_global_seeds, train_model, predict
from nlp_pipeline.pipeline import NLPPipeline

set_global_seeds(42)  # Reproducible results

## 3) Stage 1: Data Preparation


In [ ]:
# Stage 1: Text preprocessing pipeline
# Raw text → sentences → words → content words (stopword removal)
# This prepares unstructured text for further analysis (removes noise, retains signals)

text = "Cricket is popular in England. It's played worldwide! Do you follow it?"

sentences = segment_sentences(text)
tokens = tokenize_text(text)
filtered = remove_stopwords(tokens)

print('Sentences:', sentences)
print('Tokens:', tokens)
print('Stopwords removed:', filtered)

Sentences: ['Cricket is popular in England.', "It's played worldwide!", 'Do you follow it?']
Tokens: ['Cricket', 'is', 'popular', 'in', 'England', '.', 'It', "'s", 'played', 'worldwide', '!', 'Do', 'you', 'follow', 'it', '?']
Stopwords removed: ['Cricket', 'popular', 'England', '.', 'played', 'worldwide', '!', 'follow', '?']


## 4) Stage 2: AI Prepping


In [ ]:
# Stage 2: Linguistic normalization and feature extraction
# Transform tokens to base forms (running→run), extract grammar (POS) and entity types
# These normalized representations reduce sparsity and enable better semantic matching

lemmas = lemmatize_text(text)
stemmed = stem_tokens(tokens)
normalized = normalize_text(text, use_stemming=False)
pos_tags = pos_tag_text(text)
entities = named_entities(text)

print('Lemmas:', lemmas)
print('Stemmed:', stemmed)
print('Normalized:', normalized)
print('POS tags:', pos_tags)
print('Entities:', entities)

Lemmas: ['cricket', 'be', 'popular', 'in', 'England', '.', 'it', 'be', 'play', 'worldwide', '!', 'do', 'you', 'follow', 'it', '?']
Stemmed: ['cricket', 'is', 'popular', 'in', 'england', '.', 'it', "'s", 'play', 'worldwid', '!', 'do', 'you', 'follow', 'it', '?']
Normalized: ['cricket', 'be', 'popular', 'in', 'England', '.', 'it', 'be', 'play', 'worldwide', '!', 'do', 'you', 'follow', 'it', '?']
POS tags: [('Cricket', 'NOUN'), ('is', 'AUX'), ('popular', 'ADJ'), ('in', 'ADP'), ('England', 'PROPN'), ('.', 'PUNCT'), ('It', 'PRON'), ("'s", 'AUX'), ('played', 'VERB'), ('worldwide', 'ADV'), ('!', 'PUNCT'), ('Do', 'AUX'), ('you', 'PRON'), ('follow', 'VERB'), ('it', 'PRON'), ('?', 'PUNCT')]
Entities: [('England', 'GPE')]


## 5) Stage 3: AI Training (TF-IDF + Keras)


In [ ]:
# Stage 3: Machine learning classifier training
# TF-IDF converts text→numeric vectors; neural net learns to classify documents by domain
# Demonstrates supervised learning pipeline (not used in mentor matching strategy)

train_texts = [
    'great cricket match tonight',
    'fantastic football team win',
    'stock market crash and losses',
    'investors fear bad economy',
    'excellent batting and bowling',
    'market decline hurts traders',
]
train_labels = ['sports', 'sports', 'finance', 'finance', 'sports', 'finance']

vectorizer = TfidfTextVectorizer()
X_train = vectorizer.fit_transform(train_texts)
model, label_encoder = train_model(X_train, train_labels, epochs=15, batch_size=2, seed=42)
setattr(model, 'label_encoder', label_encoder)

X_new = vectorizer.transform(['cricket team won', 'economy market losses'])
preds = predict(model, X_new)
print('Predictions:', preds)

2026-02-18 10:04:06.734414: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Predictions: [np.str_('sports'), np.str_('finance')]


## 6) End-to-End Pipeline Usage


In [ ]:
# Stage 4: End-to-end pipeline demonstration
# Single interface combining all preprocessing steps + optional training
# Shows how pipeline can be reused for different text without reimplementing stages

pipeline = NLPPipeline(use_stemming=False)

sample = 'Cricket is popular in England. Fans love the sport.'
prepared = pipeline.preprocess(sample)
print('Preprocess keys:', list(prepared.keys()))
print('Entities:', prepared['entities'])

pipeline.train_classifier(train_texts, train_labels)
print('Pipeline predictions:', pipeline.predict_labels(['cricket batting skills', 'market drops again']))

Preprocess keys: ['sentences', 'tokens', 'filtered_tokens', 'normalized_tokens', 'pos_tags', 'entities']
Entities: [('England', 'GPE')]
Pipeline predictions: [np.str_('sports'), np.str_('finance')]


## 7) Test Commands
From repo root, either command works after the root `pytest.ini` fix:
- `pytest -q`
- `pytest nlp_project/tests -q`


In [ ]:
# Validate pipeline implementation
# Runs unit tests covering each processing stage + integration tests for correctness

!pytest -q

.............                                                            [100%]
=============================== warnings summary ===============================
tests/test_6_vectorize_and_tf_model.py::test_vectorize_and_train_tf_model
tests/test_6_vectorize_and_tf_model.py::test_vectorize_and_train_tf_model
tests/test_6_vectorize_and_tf_model.py::test_vectorize_and_train_tf_model
tests/test_6_vectorize_and_tf_model.py::test_vectorize_and_train_tf_model
tests/test_7_integration_pipeline.py::test_full_integration_pipeline
tests/test_7_integration_pipeline.py::test_full_integration_pipeline
tests/test_7_integration_pipeline.py::test_full_integration_pipeline
tests/test_7_integration_pipeline.py::test_full_integration_pipeline
  /home/connor/ECE495_IndepStudy/Content/Code/.venv/lib/python3.13/site-packages/keras/src/backend/tensorflow/core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and '

## 8) Strategy A: No-Training Mentor-Mentee Matching (TF-IDF + Cosine Similarity)

This section implements **Strategy A** end-to-end with no model training:
1. Load CSV data (or build richer sample mentor/mentee CSVs from `scratch.csv`)
2. Build tagged profile strings
3. Preprocess profiles consistently
4. Vectorize mentor profiles with TF-IDF
5. Rank mentors for each mentee by cosine similarity


In [ ]:
# Load or generate mentor-mentee datasets
# Attempts to load from existing CSV files; falls back to generating deterministic samples from scratch.csv
# Establishes foundation for matching algorithm (mentors and mentees with domain attributes)

from pathlib import Path
import pandas as pd
import re


def resolve_data_dir() -> Path:
    for candidate in (Path("data"), Path("../data")):
        if candidate.exists():
            return candidate
    fallback = Path("data")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def resolve_scratch_path() -> Path | None:
    candidates = [
        Path("nlp_project/scratch.csv"),
        Path("scratch.csv"),
        Path("../nlp_project/scratch.csv"),
    ]
    for c in candidates:
        if c.exists():
            return c
    return None


def load_nonempty_csv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    return df if not df.empty else None


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def build_samples_from_scratch(scratch_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_csv(scratch_path).fillna("")

    col_prior = "If yes, please explain "
    col_why = "Why are you interested in becoming an ECE Alumni Mentor?"
    col_pro = "Please talk about your professional experiences - field of interest, previous jobs or career changes, etc. "
    col_personal = "Tell us a little bit about yourself - personal interests, hobbies, family, etc. "

    # Domain taxonomy: maps keywords to standardized domain categories
    # Enables consistent tagging across mentor/mentee profiles (e.g., "firmware", "microcontroller"→"embedded systems")
    # This binning is critical for matching: mentors with diverse vocabulary still cluster by domain
    keyword_map = {
        "embedded systems": ["embedded", "firmware", "microcontroller", "rtos"],
        "power systems": ["power", "utility", "grid", "relay", "substation", "arc flash", "renewable"],
        "software engineering": ["software", "developer", "devops", "backend", "frontend", "java", "python", "cloud"],
        "machine learning": ["machine learning", "ml", "ai", "data science", "analytics", "nlp"],
        "cybersecurity": ["cyber", "security", "infosec"],
        "communications/rf": ["wireless", "rf", "telecom", "communications", "signal processing"],
        "robotics/controls": ["robotics", "control systems", "automation", "autonomous"],
        "hardware/semiconductors": ["asic", "fpga", "semiconductor", "hardware", "vlsi"],
        "product/startups": ["startup", "product", "entrepreneur", "founder"],
        "leadership": ["manager", "lead", "director", "executive", "supervisor"],
    }

    availability_slots = [
        "Mon evenings", "Tue evenings", "Wed afternoons", "Thu afternoons", "Fri mornings", "Weekends"
    ]

    def detect_domains(text: str) -> list[str]:
        t = clean_text(text).lower()
        domains = []
        for domain, kws in keyword_map.items():
            if any(k in t for k in kws):
                domains.append(domain)
        return domains

    def infer_role(professional_text: str) -> str:
        t = clean_text(professional_text).lower()
        if any(k in t for k in ["manager", "director", "executive", "supervisor", "lead"]):
            return "Engineering Leader"
        if any(k in t for k in ["professor", "phd", "research", "laboratory", "lab"]):
            return "Research Engineer"
        if any(k in t for k in ["power", "utility", "grid", "relay", "substation"]):
            return "Power Systems Engineer"
        if any(k in t for k in ["embedded", "firmware", "microcontroller", "rtos"]):
            return "Embedded Systems Engineer"
        if any(k in t for k in ["wireless", "rf", "telecom", "communications"]):
            return "Communications Engineer"
        if any(k in t for k in ["software", "developer", "devops", "backend", "frontend", "cloud", "java", "python"]):
            return "Software Engineer"
        return "ECE Professional"

    def summarize_skills(professional_text: str, domains: list[str]) -> str:
        t = clean_text(professional_text)
        phrases = list(domains[:4]) if domains else ["engineering practice", "career development"]
        first_sentence = t.split(".")[0].strip()
        if first_sentence:
            phrases.append(first_sentence[:180])
        return ", ".join(dict.fromkeys(phrases))

    mentor_rows = []
    for i, row in raw.iterrows():
        prior = clean_text(row.get(col_prior, ""))
        why = clean_text(row.get(col_why, ""))
        pro = clean_text(row.get(col_pro, ""))
        personal = clean_text(row.get(col_personal, ""))

        if not (prior or why or pro or personal):
            continue

        domains = detect_domains(f"{pro} {why}")
        mentor_rows.append(
            {
                "mentor_id": f"M{i+1:03d}",
                "mentor_name": f"Mentor {i+1:03d}",
                "role": infer_role(pro),
                "expertise": summarize_skills(pro, domains),
                "goals": why if why else "Support mentees and give back to ECE students.",
                "topics": ", ".join(domains) if domains else "career growth, engineering skills",
                "style": "experienced mentor, collaborative" if prior else "collaborative, supportive",
                "availability": availability_slots[i % len(availability_slots)],
                "bio": personal,
                "prior_mentoring_experience": prior,
                "mentorship_motivation": why,
                "professional_experience": pro,
                "personal_interests": personal,
                "domain_tags": "|".join(domains),
                "source_row_index": int(i),
            }
        )

    mentors = pd.DataFrame(mentor_rows)
    if len(mentors) < 5:
        raise ValueError("Unable to parse enough mentor records from scratch.csv")

    mentee_rows = []
    num_mentees = min(40, len(mentors))
    for j in range(num_mentees):
        mentor = mentors.iloc[j]
        domains = [d for d in str(mentor.get("domain_tags", "")).split("|") if d]
        interests = ", ".join(domains[:3]) if domains else str(mentor.get("topics", "engineering skills"))
        goals = (
            f"Learn practical guidance in {domains[0]} and prepare for internships/projects."
            if domains else
            "Learn practical engineering and career skills from an experienced mentor."
        )

        mentee_rows.append(
            {
                "mentee_id": f"T{j+1:03d}",
                "mentee_name": f"Mentee {j+1:03d}",
                "role": "ECE Student",
                "interests": interests,
                "goals": goals,
                "topics": str(mentor.get("topics", "career growth")),
                "style": "hands-on, structured",
                "availability": str(mentor.get("availability", availability_slots[j % len(availability_slots)])),
                "background": f"Interested in {interests}. {str(mentor.get('personal_interests', ''))[:140]}".strip(),
                "source_mentor_id": str(mentor.get("mentor_id")),
                "source_row_index": int(mentor.get("source_row_index", j)),
            }
        )

    mentees = pd.DataFrame(mentee_rows)
    return mentors, mentees


def ensure_identity_columns(
    df: pd.DataFrame,
    id_col: str,
    name_col: str,
    id_candidates: list[str],
    name_candidates: list[str],
    prefix: str,
) -> pd.DataFrame:
    df = df.copy()

    source_id = next((c for c in id_candidates if c in df.columns), None)
    if source_id:
        df[id_col] = df[source_id].astype(str).str.strip()
    elif id_col not in df.columns:
        df[id_col] = [f"{prefix}{i+1:03d}" for i in range(len(df))]

    source_name = next((c for c in name_candidates if c in df.columns), None)
    if source_name:
        df[name_col] = df[source_name].astype(str).str.strip()
    elif name_col not in df.columns:
        df[name_col] = [f"{prefix.lower()}_{i+1:03d}" for i in range(len(df))]

    fallback_ids = pd.Series([f"{prefix}{i+1:03d}" for i in range(len(df))], index=df.index)
    fallback_names = pd.Series([f"{prefix.lower()}_{i+1:03d}" for i in range(len(df))], index=df.index)
    df[id_col] = df[id_col].replace({"": pd.NA}).fillna(fallback_ids)
    df[name_col] = df[name_col].replace({"": pd.NA}).fillna(fallback_names)
    return df


DATA_DIR = resolve_data_dir()
MENTEES_PATH = DATA_DIR / "sample_mentees.csv"
MENTORS_PATH = DATA_DIR / "sample_mentors.csv"
SCRATCH_PATH = resolve_scratch_path()

FORCE_REBUILD_FROM_SCRATCH = False
ALWAYS_WRITE_SAMPLE_CSV = True

mentees_df = None if FORCE_REBUILD_FROM_SCRATCH else load_nonempty_csv(MENTEES_PATH)
mentors_df = None if FORCE_REBUILD_FROM_SCRATCH else load_nonempty_csv(MENTORS_PATH)
used_synthetic_data = False

if (mentees_df is None or mentors_df is None) and SCRATCH_PATH is not None:
    mentors_df, mentees_df = build_samples_from_scratch(SCRATCH_PATH)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    mentors_df.to_csv(MENTORS_PATH, index=False)
    mentees_df.to_csv(MENTEES_PATH, index=False)
    used_synthetic_data = True
    print(f"Generated samples from {SCRATCH_PATH} -> {MENTORS_PATH} and {MENTEES_PATH}")
elif mentees_df is None or mentors_df is None:
    raise FileNotFoundError(
        "Missing sample CSVs and no scratch.csv found. Please add scratch.csv or sample CSV files."
    )
else:
    print(f"Loaded existing sample data from: {MENTORS_PATH} and {MENTEES_PATH}")

mentors_df = ensure_identity_columns(
    mentors_df,
    id_col="mentor_id",
    name_col="mentor_name",
    id_candidates=["mentor_id", "Mentor ID", "id", "Email Address"],
    name_candidates=["mentor_name", "First & Last Name", "name"],
    prefix="M",
)
mentees_df = ensure_identity_columns(
    mentees_df,
    id_col="mentee_id",
    name_col="mentee_name",
    id_candidates=["mentee_id", "Mentee ID", "id", "Email Address"],
    name_candidates=["mentee_name", "First & Last Name", "name"],
    prefix="T",
)

if ALWAYS_WRITE_SAMPLE_CSV:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    mentors_df.to_csv(MENTORS_PATH, index=False)
    mentees_df.to_csv(MENTEES_PATH, index=False)
    print(f"Persisted CSVs to disk: {MENTORS_PATH} and {MENTEES_PATH}")

print(f"Mentors: {len(mentors_df)} | Mentees: {len(mentees_df)}")
display(mentors_df.head(3))
display(mentees_df.head(3))

Loaded existing sample data from: ../data/sample_mentors.csv and ../data/sample_mentees.csv
Persisted CSVs to disk: ../data/sample_mentors.csv and ../data/sample_mentees.csv
Mentors: 131 | Mentees: 40


,mentor_id,mentor_name,role,expertise,goals,topics,style,availability,bio,prior_mentoring_experience,mentorship_motivation,professional_experience,personal_interests,domain_tags,source_row_index
0,M001,Mentor 001,Power Systems Engineer,"power systems, I am a registered Electrical En...",Support mentees and give back to ECE students.,power systems,"collaborative, supportive",Mon evenings,NaN,NaN,NaN,I am a registered Electrical Engineer in seven...,NaN,power systems,0
1,M002,Mentor 002,Engineering Leader,"product/startups, leadership, I've worked in s...",I wish that I was connected to more alumni whe...,"product/startups, leadership","experienced mentor, collaborative",Tue evenings,Married and new father. Play ice hockey.,Currently a mentor in the NCSU Entrepreneurs P...,I wish that I was connected to more alumni whe...,I've worked in startups and bigger companies. ...,Married and new father. Play ice hockey.,product/startups|leadership,1
2,M003,Mentor 003,Research Engineer,"power systems, communications/rf, hardware/sem...",I love being able to give back to my Alma Mate...,"power systems, communications/rf, hardware/sem...","experienced mentor, collaborative",Wed afternoons,I am a huge sports fan and love to be active i...,"Mentoring: MIT Undergraduate Students, MIT Lin...",I love being able to give back to my Alma Mate...,I have been an employee of MIT Lincoln Laborat...,I am a huge sports fan and love to be active i...,power systems|communications/rf|hardware/semic...,2


,mentee_id,mentee_name,role,interests,goals,topics,style,availability,background,source_mentor_id,source_row_index
0,T001,Mentee 001,ECE Student,power systems,Learn practical guidance in power systems and ...,power systems,"hands-on, structured",Mon evenings,Interested in power systems.,M001,0
1,T002,Mentee 002,ECE Student,"product/startups, leadership",Learn practical guidance in product/startups a...,"product/startups, leadership","hands-on, structured",Tue evenings,"Interested in product/startups, leadership. Ma...",M002,1
2,T003,Mentee 003,ECE Student,"power systems, communications/rf, hardware/sem...",Learn practical guidance in power systems and ...,"power systems, communications/rf, hardware/sem...","hands-on, structured",Wed afternoons,"Interested in power systems, communications/rf...",M003,2


In [ ]:
# Construct unified profile representation for matching
# Converts CSV columns (role, skills, goals, etc.) into pipe-delimited strings with semantic tags  
# This structured format becomes input to vectorizer for similarity computation

from typing import Iterable, Optional
import pandas as pd


ROLE_MENTOR_CANDIDATES = ["role", "mentor_role", "title"]
ROLE_MENTEE_CANDIDATES = ["role", "mentee_role", "student_role"]

EXPERTISE_CANDIDATES = ["expertise", "skills", "specialties", "professional_experience"]
INTERESTS_CANDIDATES = ["interests", "interest_areas", "learning_interests", "domain_tags"]
GOALS_CANDIDATES = ["goals", "objective", "mentorship_goals", "mentorship_motivation"]
TOPICS_CANDIDATES = ["topics", "focus_topics", "areas", "domain_tags"]
STYLE_CANDIDATES = ["style", "mentoring_style", "preferred_style", "prior_mentoring_experience"]
AVAILABILITY_CANDIDATES = ["availability", "available_times", "schedule"]
BIO_CANDIDATES = ["bio", "background", "notes", "personal_interests", "professional_experience"]


def first_nonempty_value(row: pd.Series, candidates: Iterable[str], default: str = "") -> str:
    for col in candidates:
        value = row.get(col, "")
        if pd.notna(value):
            text = str(value).strip()
            if text:
                return text
    return default


def build_mentor_profile(row: pd.Series, extra_fields: Optional[Iterable[str]] = None) -> str:
    parts = [
        f"ROLE: {first_nonempty_value(row, ROLE_MENTOR_CANDIDATES, 'mentor')}",
        f"EXPERTISE: {first_nonempty_value(row, EXPERTISE_CANDIDATES, 'general mentoring')}",
        f"GOALS: {first_nonempty_value(row, GOALS_CANDIDATES, 'support mentees')}",
        f"TOPICS: {first_nonempty_value(row, TOPICS_CANDIDATES, 'professional development')}",
        f"STYLE: {first_nonempty_value(row, STYLE_CANDIDATES, 'flexible')}",
        f"AVAILABILITY: {first_nonempty_value(row, AVAILABILITY_CANDIDATES, 'not specified')}",
        f"BIO: {first_nonempty_value(row, BIO_CANDIDATES, '')}",
    ]
    if extra_fields:
        for field in extra_fields:
            value = row.get(field, "")
            if pd.notna(value) and str(value).strip():
                parts.append(f"{field.upper()}: {str(value).strip()}")
    return " | ".join(parts)


def build_mentee_profile(row: pd.Series, extra_fields: Optional[Iterable[str]] = None) -> str:
    parts = [
        f"ROLE: {first_nonempty_value(row, ROLE_MENTEE_CANDIDATES, 'mentee')}",
        f"INTERESTS: {first_nonempty_value(row, INTERESTS_CANDIDATES, 'general growth')}",
        f"GOALS: {first_nonempty_value(row, GOALS_CANDIDATES, 'learn from mentor')}",
        f"TOPICS: {first_nonempty_value(row, TOPICS_CANDIDATES, 'career development')}",
        f"STYLE: {first_nonempty_value(row, STYLE_CANDIDATES, 'flexible')}",
        f"AVAILABILITY: {first_nonempty_value(row, AVAILABILITY_CANDIDATES, 'not specified')}",
        f"BACKGROUND: {first_nonempty_value(row, BIO_CANDIDATES, '')}",
    ]
    if extra_fields:
        for field in extra_fields:
            value = row.get(field, "")
            if pd.notna(value) and str(value).strip():
                parts.append(f"{field.upper()}: {str(value).strip()}")
    return " | ".join(parts)


mentor_extra_fields = [
    c for c in ("industry", "location", "domain_tags", "professional_experience", "mentorship_motivation")
    if c in mentors_df.columns
]
mentee_extra_fields = [
    c for c in ("career_stage", "location", "source_mentor_id")
    if c in mentees_df.columns
]

mentors_df = mentors_df.copy()
mentees_df = mentees_df.copy()
mentors_df["mentor_profile_raw"] = mentors_df.apply(
    lambda row: build_mentor_profile(row, extra_fields=mentor_extra_fields),
    axis=1,
)
mentees_df["mentee_profile_raw"] = mentees_df.apply(
    lambda row: build_mentee_profile(row, extra_fields=mentee_extra_fields),
    axis=1,
)

display_cols_mentor = [c for c in ["mentor_id", "mentor_name", "mentor_profile_raw"] if c in mentors_df.columns]
display_cols_mentee = [c for c in ["mentee_id", "mentee_name", "mentee_profile_raw"] if c in mentees_df.columns]

display(mentors_df[display_cols_mentor].head(2))
display(mentees_df[display_cols_mentee].head(2))

,mentor_id,mentor_name,mentor_profile_raw
0,M001,Mentor 001,ROLE: Power Systems Engineer | EXPERTISE: powe...
1,M002,Mentor 002,ROLE: Engineering Leader | EXPERTISE: product/...


,mentee_id,mentee_name,mentee_profile_raw
0,T001,Mentee 001,ROLE: ECE Student | INTERESTS: power systems |...
1,T002,Mentee 002,ROLE: ECE Student | INTERESTS: product/startup...


In [ ]:
# Normalize and cleanse profile text for vectorization
# Removes noise (punctuation, special chars), applies stopword removal, lemmatizes with spaCy if available
# Fallback chain: nlp_pipeline → spaCy → local regex (ensures robustness even with missing deps)

import re
import pandas as pd
from typing import Optional

USE_STOPWORD_REMOVAL = True
BASE_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "in",
    "is", "it", "of", "on", "or", "that", "the", "to", "with", "this", "these",
    "those", "i", "we", "you", "they", "our", "their", "my", "your", "not"
}


def local_preprocess_text(text: Optional[str], remove_stopwords: bool = USE_STOPWORD_REMOVAL) -> str:
    text = "" if text is None else str(text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s+/#-]", " ", text)
    text = re.sub(r"[|:,;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if remove_stopwords and text:
        tokens = [tok for tok in text.split() if tok not in BASE_STOPWORDS]
        text = " ".join(tokens)
    return text


PIPELINE_BACKEND = "local_preprocess"
_pipeline = None

try:
    from nlp_pipeline.pipeline import NLPPipeline
    _pipeline = NLPPipeline()

    def pipeline_preprocess_text(text: Optional[str]) -> str:
        result = _pipeline.preprocess("" if text is None else str(text))
        for key in ("normalized_tokens", "filtered_tokens", "tokens"):
            value = result.get(key)
            if isinstance(value, list) and value:
                return " ".join(str(tok) for tok in value if str(tok).strip())
        return local_preprocess_text(text)

    PIPELINE_BACKEND = "nlp_pipeline.NLPPipeline.preprocess"
except Exception as pipeline_exc:
    PIPELINE_BACKEND = f"local_preprocess (pipeline unavailable: {pipeline_exc.__class__.__name__})"


SPACY_AVAILABLE = False
SPACY_ENGINE = "none"
try:
    import spacy

    try:
        spacy_nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
        SPACY_ENGINE = "en_core_web_sm"
    except Exception:
        spacy_nlp = spacy.blank("en")
        if "sentencizer" not in spacy_nlp.pipe_names:
            spacy_nlp.add_pipe("sentencizer")
        SPACY_ENGINE = "spacy.blank('en')"

    SPACY_AVAILABLE = True
except Exception:
    spacy_nlp = None


# Advanced preprocessing: lemmatization + POS-aware stopword removal (spaCy)
# Base forms reduce feature sparsity; POS-tagging helps exclude noise while preserving meaningful words
def spacy_preprocess_text(text: Optional[str]) -> str:
    if not SPACY_AVAILABLE or spacy_nlp is None:
        return local_preprocess_text(text)

    doc = spacy_nlp("" if text is None else str(text))
    tokens = []
    for tok in doc:
        raw = tok.lemma_ if tok.lemma_ not in ("", "-PRON-") else tok.text
        clean = raw.lower().strip()
        if not clean:
            continue
        if not re.search(r"[a-z0-9]", clean):
            continue
        if USE_STOPWORD_REMOVAL and (clean in BASE_STOPWORDS or getattr(tok, "is_stop", False)):
            continue
        tokens.append(clean)
    return " ".join(tokens) if tokens else local_preprocess_text(text)


# Route preprocessing through best available backend
# Tries: nlp_pipeline (if available) → spaCy lemmatization (better quality) → local fallback (always works)
def preprocess_profile(text: Optional[str]) -> str:
    if _pipeline is not None:
        try:
            return local_preprocess_text(pipeline_preprocess_text(text))
        except Exception:
            pass
    if SPACY_AVAILABLE:
        return spacy_preprocess_text(text)
    return local_preprocess_text(text)


mentors_df["mentor_profile"] = mentors_df["mentor_profile_raw"].map(preprocess_profile)
mentees_df["mentee_profile"] = mentees_df["mentee_profile_raw"].map(preprocess_profile)

print("Preprocess backend:", PIPELINE_BACKEND)
print("spaCy engine:", SPACY_ENGINE)
display(mentors_df[["mentor_id", "mentor_profile"]].head(2))
display(mentees_df[["mentee_id", "mentee_profile"]].head(2))

Preprocess backend: nlp_pipeline.NLPPipeline.preprocess
spaCy engine: en_core_web_sm


,mentor_id,mentor_profile
0,M001,role power systems engineer expertise power sy...
1,M002,role engineering leader expertise product / st...


,mentee_id,mentee_profile
0,T001,role ece student interests power system goals ...
1,T002,role ece student interests product / startup l...


In [ ]:
# Optional: Hard constraints gate before similarity ranking (disabled by default)
# Allows filtering mentees/mentors by domain, time, location before vectorization
# Extension point for business rules (e.g., "only match ECE majors and available mentors")

import pandas as pd
from typing import Tuple

def apply_optional_hard_filters(
    mentees: pd.DataFrame, mentors: pd.DataFrame, enable: bool = False
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not enable:
        return mentees, mentors

    try:
        import filters as hard_filters
        if hasattr(hard_filters, "apply_filters"):
            print("filters.apply_filters found, but current signature is not DataFrame-based; skipping.")
        else:
            print("No compatible filter function found; skipping.")
    except Exception as exc:
        print(f"Optional filters unavailable ({exc.__class__.__name__}); skipping.")
    return mentees, mentors


ENABLE_HARD_FILTERS = False
mentees_for_match, mentors_for_match = apply_optional_hard_filters(
    mentees_df,
    mentors_df,
    enable=ENABLE_HARD_FILTERS,
)

In [ ]:
# Vectorize profiles with TF-IDF and compute mentor-mentee alignment scores
# TF-IDF captures what topics each profile emphasizes; cosine similarity measures directional overlap
# Result: 40×40 matrix where element [i,j] = similarity(mentee_i, mentor_j) ∈ [0,1]

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TOP_K = 3

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    lowercase=True,
)

# Fit vocabulary on mentors (establishes what terms matter), then vectorize both groups
mentor_matrix = vectorizer.fit_transform(mentors_for_match["mentor_profile"])
mentee_matrix = vectorizer.transform(mentees_for_match["mentee_profile"])
# Score pairwise similarity (higher = more aligned interests/goals)
similarity_matrix = cosine_similarity(mentee_matrix, mentor_matrix)

feature_names = vectorizer.get_feature_names_out()


def overlap_terms(mentee_text: str, mentor_text: str, limit: int = 5) -> str:
    mentee_terms = set(mentee_text.split())
    mentor_terms = set(mentor_text.split())
    overlap = sorted(mentee_terms.intersection(mentor_terms))
    return ", ".join(overlap[:limit])


def get_top_k_matches(
    mentees: pd.DataFrame,
    mentors: pd.DataFrame,
    sim_matrix: np.ndarray,
    k: int = TOP_K,
) -> pd.DataFrame:
    rows: list[dict] = []
    effective_k = min(k, len(mentors))

    for mentee_idx, mentee_row in mentees.reset_index(drop=True).iterrows():
        scores: np.ndarray = sim_matrix[mentee_idx]
        top_indices = np.argsort(scores)[::-1][:effective_k]

        for rank, mentor_idx in enumerate(top_indices, start=1):
            mentor_row = mentors.reset_index(drop=True).iloc[mentor_idx]
            mentee_id = mentee_row.get("mentee_id", f"mentee_{mentee_idx}")
            mentee_name = mentee_row.get("mentee_name", "Unknown Mentee")
            mentor_id = mentor_row.get("mentor_id", f"mentor_{mentor_idx}")
            mentor_name = mentor_row.get("mentor_name", "Unknown Mentor")
            rows.append(
                {
                    "mentee_id": mentee_id,
                    "mentee_name": mentee_name,
                    "mentor_id": mentor_id,
                    "mentor_name": mentor_name,
                    "score": float(scores[mentor_idx]),
                    "rank": rank,
                    "overlap_terms": overlap_terms(
                        mentee_row.get("mentee_profile", ""),
                        mentor_row.get("mentor_profile", ""),
                    ),
                }
            )

    return pd.DataFrame(rows)


top_matches_df = get_top_k_matches(
    mentees_for_match,
    mentors_for_match,
    similarity_matrix,
    k=TOP_K,
)

columns_order = [
    "mentee_id", "mentee_name", "mentor_id", "mentor_name",
    "score", "rank", "overlap_terms",
]
top_matches_df = top_matches_df[columns_order].sort_values(
    by=["mentee_id", "rank"],
    ascending=[True, True],
).reset_index(drop=True)

In [ ]:
# Quality assurance: verify output integrity and algorithmic correctness
# Checks: matrix dimensions consistent with input, scores in valid range [0,1], top-1 alignment with seed data
# Catches silent failures (e.g., vectorization bugs, rank misalignment, data corruption)

print(f"Mentor-mentee similarity matrix shape: {similarity_matrix.shape}")
print(f"Expected: ({len(mentees)}, {len(mentors)})")
print(f"Score range: min={similarity_matrix.min():.4f}, max={similarity_matrix.max():.4f}")
assert (similarity_matrix >= 0).all() and (similarity_matrix <= 1).all()
print("✓ All scores in valid range [0,1]")

print(f"\nTop matches dataset size: {len(top_matches_df)} rows (expecting ≤ {len(mentees) * k})")
mentee_counts = top_matches_df.groupby("mentee_id").size()
print(f"Matches per mentee: min={mentee_counts.min()}, max={mentee_counts.max()}")
assert (top_matches_df["rank"] >= 1).all() and (top_matches_df["rank"] <= k).all()
print(f"✓ All ranks in [1, {k}]")


Mentee 001 (T001)
  #1 -> Mentor 106 (M106), score=0.2948, overlap=[-, availability, goals, mentor, power]
  #2 -> Mentor 045 (M045), score=0.2398, overlap=[availability, goals, learn, mentor, power]
  #3 -> Mentor 067 (M067), score=0.2192, overlap=[-, availability, evening, goals, mentor]

Mentee 002 (T002)
  #1 -> Mentor 002 (M002), score=0.3954, overlap=[/, availability, ece, evening, father]
  #2 -> Mentor 099 (M099), score=0.2601, overlap=[/, availability, mentor, product, role]
  #3 -> Mentor 049 (M049), score=0.2082, overlap=[-, /, availability, ece, evening]

Mentee 003 (T003)
  #1 -> Mentor 003 (M003), score=0.3688, overlap=[--, /, active, afternoon, availability]
  #2 -> Mentor 026 (M026), score=0.2935, overlap=[/, availability, communication, fan, find]
  #3 -> Mentor 105 (M105), score=0.2477, overlap=[/, afternoon, availability, communication, d]

Mentee 004 (T004)
  #1 -> Mentor 004 (M004), score=0.3080, overlap=[1, afternoon, athletics, availability, beer]
  #2 -> Mentor

,mentee_id,mentee_name,mentor_id,mentor_name,score,rank,overlap_terms
0,T001,Mentee 001,M106,Mentor 106,0.294753,1,"-, availability, goals, mentor, power"
1,T001,Mentee 001,M045,Mentor 045,0.239833,2,"availability, goals, learn, mentor, power"
2,T001,Mentee 001,M067,Mentor 067,0.219231,3,"-, availability, evening, goals, mentor"
3,T002,Mentee 002,M002,Mentor 002,0.395351,1,"/, availability, ece, evening, father"
4,T002,Mentee 002,M099,Mentor 099,0.260069,2,"/, availability, mentor, product, role"
...,...,...,...,...,...,...,...
115,T039,Mentee 039,M077,Mentor 077,0.115710,2,"availability, goals, husband, leadership, learn"
116,T039,Mentee 039,M100,Mentor 100,0.101545,3,"afternoon, availability, leadership, mentor, role"
117,T040,Mentee 040,M040,Mentor 040,0.356448,1,"afternoon, availability, chennai, degree, embed"
118,T040,Mentee 040,M061,Mentor 061,0.205009,2,"/, availability, embed, finish, goals"


In [ ]:
# Quality assurance: verify output integrity and algorithmic correctness
# Checks: matrix dimensions consistent, scores are valid (finite/non-negative), top-1 matches align with seed data

expected_k = min(TOP_K, len(mentors_for_match))

assert mentor_matrix.shape[0] == len(mentors_for_match), "Mentor vector row count mismatch"
assert similarity_matrix.shape == (len(mentees_for_match), len(mentors_for_match)), "Similarity matrix shape mismatch"
assert top_matches_df.groupby("mentee_id").size().eq(expected_k).all(), "Each mentee must have K matches"
assert np.isfinite(top_matches_df["score"]).all(), "Scores must be finite"
assert (top_matches_df["score"] >= 0).all(), "Scores must be non-negative"
# Cross-validate: do top-ranked mentors match the original mentee-mentor pairings?
# Check if top-1 matches align with source mentor IDs (when available)
if "source_mentor_id" in mentees_for_match.columns:
    top1 = top_matches_df[top_matches_df["rank"] == 1]
    expected = mentees_for_match[["mentee_id", "source_mentor_id"]].copy()
    merged = top1.merge(expected, on="mentee_id", how="left")
    aligned = (merged["mentor_id"] == merged["source_mentor_id"]).fillna(False)
    alignment_rate = float(aligned.mean()) if len(aligned) else 0.0
    assert aligned.sum() >= 1, "Expected at least one source mentor to appear as top-1"
    print(f"Top-1 alignment with source_mentor_id: {alignment_rate:.1%}")

print("All Strategy A sanity checks passed.")

Top-1 alignment with source_mentor_id: 85.0%
All Strategy A sanity checks passed.


## 9) How To Use With Real Data

- Expected CSV columns (minimum):
  - Mentors: `mentor_id`, `mentor_name`, plus any of `role`, `expertise`, `goals`, `topics`, `style`, `availability`, `bio`
  - Mentees: `mentee_id`, `mentee_name`, plus any of `role`, `interests`, `goals`, `topics`, `style`, `availability`, `background`
- Working from `scratch.csv`:
  - Place it at `nlp_project/scratch.csv` (or adjust path candidates in the load cell).
  - If sample CSVs are missing, the notebook will auto-generate `data/sample_mentors.csv` and `data/sample_mentees.csv` from `scratch.csv`.
- To force regeneration each run:
  - Set `FORCE_REBUILD_FROM_SCRATCH = True` in the load cell.
- To adjust profile schema:
  - Edit `build_mentor_profile(...)` and `build_mentee_profile(...)` tag/field candidates.
- To add hard filters before ranking:
  - Implement DataFrame-compatible rules in `filters.py`, then set `ENABLE_HARD_FILTERS = True`.
